In [2]:
import pandas as pd
import numpy as np

print("Market Basket Analysis Project")
print("Libraries loaded successfully!")


Market Basket Analysis Project
Libraries loaded successfully!


In [3]:
df = pd.read_excel("../data/raw/Online Retail.xlsx")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (541909, 8)


In [4]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [6]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [7]:
# Check cancelled transactions

cancelled = df[df['InvoiceNo'].astype(str).str.startswith('C')]

print("Cancelled rows:", len(cancelled))
print("Percentage:", round(len(cancelled) / len(df) * 100, 2), "%")

Cancelled rows: 9288
Percentage: 1.71 %


In [8]:
# Check negative quantities

negative_quantity = df[df['Quantity'] < 0]

print("Negative quantity rows:", len(negative_quantity))

Negative quantity rows: 10624


In [9]:
# Check duplicate rows

print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 5268


In [10]:
print("Zero price rows:", (df['UnitPrice'] == 0).sum())
print("Negative price rows:", (df['UnitPrice'] < 0).sum())

Zero price rows: 2515
Negative price rows: 2


In [11]:
print("Missing descriptions:", df['Description'].isnull().sum())

Missing descriptions: 1454


In [12]:
print("Unique products:", df['StockCode'].nunique())
print("Unique product names:", df['Description'].nunique())

Unique products: 4070
Unique product names: 4223


In [13]:
clean_df = df.copy()

print("Original rows:", len(clean_df))

Original rows: 541909


In [14]:
# 1. Remove cancelled invoices
clean_df = clean_df[
    ~clean_df['InvoiceNo'].astype(str).str.startswith('C')
]

# 2. Keep only positive quantities
clean_df = clean_df[clean_df['Quantity'] > 0]

# 3. Keep only positive prices
clean_df = clean_df[clean_df['UnitPrice'] > 0]

# 4. Remove rows without product descriptions
clean_df = clean_df.dropna(subset=['Description'])

# 5. Remove duplicate rows
clean_df = clean_df.drop_duplicates()

# 6. Clean product descriptions
clean_df['Description'] = clean_df['Description'].str.strip()

print("Cleaning completed!")
print("Remaining rows:", len(clean_df))

Cleaning completed!
Remaining rows: 524878


In [15]:
import os

os.makedirs("../data/processed", exist_ok=True)

In [16]:
clean_df.to_csv(
    "../data/processed/cleaned_retail.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [17]:
print("Total transactions:", clean_df['InvoiceNo'].nunique())
print("Unique products:", clean_df['StockCode'].nunique())
print("Unique customers:", clean_df['CustomerID'].nunique())
print("Countries:", clean_df['Country'].nunique())

Total transactions: 19960
Unique products: 3922
Unique customers: 4338
Countries: 38


In [18]:
clean_df['Revenue'] = clean_df['Quantity'] * clean_df['UnitPrice']

print("Total Revenue: £", round(clean_df['Revenue'].sum(), 2))
print("Average transaction value: £",
      round(clean_df.groupby('InvoiceNo')['Revenue'].sum().mean(), 2))

Total Revenue: £ 10642110.8
Average transaction value: £ 533.17


In [19]:
top_products = (
    clean_df.groupby('Description')['Quantity']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products

Description
PAPER CRAFT , LITTLE BIRDIE           80995
MEDIUM CERAMIC TOP STORAGE JAR        78033
WORLD WAR 2 GLIDERS ASSTD DESIGNS     54951
JUMBO BAG RED RETROSPOT               48371
WHITE HANGING HEART T-LIGHT HOLDER    37872
POPCORN HOLDER                        36749
PACK OF 72 RETROSPOT CAKE CASES       36396
ASSORTED COLOUR BIRD ORNAMENT         36362
RABBIT NIGHT LIGHT                    30739
MINI PAINT SET VINTAGE                26633
Name: Quantity, dtype: int64

In [20]:
print("Total transactions:", clean_df['InvoiceNo'].nunique())
print("Unique products:", clean_df['StockCode'].nunique())
print("Unique customers:", clean_df['CustomerID'].nunique())
print("Countries:", clean_df['Country'].nunique())

Total transactions: 19960
Unique products: 3922
Unique customers: 4338
Countries: 38


In [21]:
top_revenue_products = (
    clean_df.groupby('Description')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_revenue_products

Description
DOTCOM POSTAGE                        206248.77
REGENCY CAKESTAND 3 TIER              174156.54
PAPER CRAFT , LITTLE BIRDIE           168469.60
WHITE HANGING HEART T-LIGHT HOLDER    106236.72
PARTY BUNTING                          99445.23
JUMBO BAG RED RETROSPOT                94159.81
MEDIUM CERAMIC TOP STORAGE JAR         81700.92
POSTAGE                                78101.88
Manual                                 77752.82
RABBIT NIGHT LIGHT                     66870.03
Name: Revenue, dtype: float64

In [22]:
items_per_transaction = (
    clean_df.groupby('InvoiceNo')['Quantity']
    .sum()
)

print("Average items per transaction:",
      round(items_per_transaction.mean(), 2))

print("Maximum items in one transaction:",
      items_per_transaction.max())

Average items per transaction: 279.18
Maximum items in one transaction: 80995


In [23]:
basket_stats = (
    clean_df.groupby('InvoiceNo')['StockCode']
    .nunique()
)

print("Average unique products per transaction:",
      round(basket_stats.mean(), 2))

print("Median unique products per transaction:",
      basket_stats.median())

print("Maximum unique products in one transaction:",
      basket_stats.max())

Average unique products per transaction: 26.03
Median unique products per transaction: 15.0
Maximum unique products in one transaction: 1110


In [24]:
print(basket_stats.describe())

count    19960.000000
mean        26.032164
std         46.982195
min          1.000000
25%          6.000000
50%         15.000000
75%         29.000000
max       1110.000000
Name: StockCode, dtype: float64


In [25]:
largest_baskets = (
    basket_stats
    .sort_values(ascending=False)
    .head(10)
)

largest_baskets

InvoiceNo
573585    1110
581219     749
581492     731
580729     721
558475     704
579777     687
581217     676
537434     674
580730     661
580367     650
Name: StockCode, dtype: int64

In [26]:
print("Basket size percentiles:")
print(basket_stats.quantile([0.90, 0.95, 0.99, 0.995, 0.999]))

Basket size percentiles:
0.900     52.000
0.950     75.000
0.990    217.410
0.995    341.615
0.999    590.123
Name: StockCode, dtype: float64


In [27]:
large_invoices = basket_stats[basket_stats > 100]

print("Invoices with more than 100 unique products:",
      len(large_invoices))

print("Percentage of all invoices:",
      round(len(large_invoices) / len(basket_stats) * 100, 2), "%")

Invoices with more than 100 unique products: 628
Percentage of all invoices: 3.15 %


In [28]:
large_invoice_ids = basket_stats[basket_stats > 100].index

large_basket_details = clean_df[
    clean_df['InvoiceNo'].isin(large_invoice_ids)
]

print(
    large_basket_details.groupby('Country')['InvoiceNo']
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

Country
United Kingdom    609
Netherlands         3
EIRE                3
Switzerland         3
France              2
Spain               2
Germany             2
Australia           1
Israel              1
Portugal            1
Name: InvoiceNo, dtype: int64


In [29]:
large_summary = (
    large_basket_details.groupby('InvoiceNo')
    .agg(
        unique_products=('StockCode', 'nunique'),
        total_quantity=('Quantity', 'sum'),
        revenue=('Revenue', 'sum'),
        customer=('CustomerID', 'first'),
        country=('Country', 'first')
    )
    .sort_values('unique_products', ascending=False)
)

large_summary.head(15)

,unique_products,total_quantity,revenue,customer,country
InvoiceNo,,,,,
573585,1110,5198,16874.58,NaN,United Kingdom
581219,749,2151,8159.03,NaN,United Kingdom
581492,731,2011,7689.23,NaN,United Kingdom
580729,721,2456,9020.83,NaN,United Kingdom
558475,704,4137,8705.89,NaN,United Kingdom
579777,687,1763,6769.06,NaN,United Kingdom
581217,676,1853,6947.75,NaN,United Kingdom
537434,674,1869,8223.40,NaN,United Kingdom
580730,661,1858,6870.20,NaN,United Kingdom


In [30]:
print("Missing CustomerID among large baskets:",
      large_summary['customer'].isna().sum())

print("Large baskets with CustomerID:",
      large_summary['customer'].notna().sum())

Missing CustomerID among large baskets: 421
Large baskets with CustomerID: 207


In [31]:
large_summary.describe()

,unique_products,total_quantity,revenue,customer
count,628.000000,628.000000,628.000000,207.000000
mean,223.321656,859.036624,2581.477516,15251.574879
std,140.021923,1309.145971,3326.815867,1653.916920
min,101.000000,166.000000,222.470000,12357.000000
25%,129.000000,367.000000,1137.780000,14096.000000
50%,175.500000,506.500000,1672.610000,14911.000000
75%,249.250000,914.250000,2770.837500,16686.000000
max,1110.000000,15049.000000,52940.940000,18122.000000


In [32]:
print("Basket size percentiles:")
print(
    basket_stats.quantile(
        [0.90, 0.95, 0.975, 0.99, 0.995, 0.999]
    )
)


Basket size percentiles:
0.900     52.000
0.950     75.000
0.975    122.000
0.990    217.410
0.995    341.615
0.999    590.123
Name: StockCode, dtype: float64


In [33]:
print("Number of baskets by size:")

bins = [0, 10, 25, 50, 100, 200, 500, 1000, float('inf')]

labels = [
    "1-10",
    "11-25",
    "26-50",
    "51-100",
    "101-200",
    "201-500",
    "501-1000",
    "1000+"
]

basket_categories = pd.cut(
    basket_stats,
    bins=bins,
    labels=labels
)

print(basket_categories.value_counts().sort_index())

Number of baskets by size:
StockCode
1-10        7487
11-25       6660
26-50       3711
51-100      1474
101-200      392
201-500      187
501-1000      48
1000+          1
Name: count, dtype: int64


In [34]:
# Calculate basket size
basket_size = clean_df.groupby('InvoiceNo')['StockCode'].nunique()

# Keep baskets with at most 100 unique products
valid_invoices = basket_size[basket_size <= 100].index

mba_df = clean_df[
    clean_df['InvoiceNo'].isin(valid_invoices)
].copy()

print("Original cleaned rows:", len(clean_df))
print("MBA modelling rows:", len(mba_df))
print("MBA transactions:", mba_df['InvoiceNo'].nunique())
print("Removed transactions:",
      clean_df['InvoiceNo'].nunique() - mba_df['InvoiceNo'].nunique())

Original cleaned rows: 524878
MBA modelling rows: 383555
MBA transactions: 19332
Removed transactions: 628


In [35]:
mba_basket_size = mba_df.groupby('InvoiceNo')['StockCode'].nunique()

print("Maximum basket size:",
      mba_basket_size.max())

print("Average basket size:",
      round(mba_basket_size.mean(), 2))

print("Median basket size:",
      mba_basket_size.median())

Maximum basket size: 100
Average basket size: 19.62
Median basket size: 15.0


In [36]:
# Calculate number of unique products in each basket
basket_size = clean_df.groupby('InvoiceNo')['StockCode'].nunique()

# Keep baskets with at most 100 unique products
valid_invoices = basket_size[basket_size <= 100].index

mba_df = clean_df[
    clean_df['InvoiceNo'].isin(valid_invoices)
].copy()

print("Original cleaned rows:", len(clean_df))
print("MBA modelling rows:", len(mba_df))
print("MBA transactions:", mba_df['InvoiceNo'].nunique())
print("Removed transactions:",
      clean_df['InvoiceNo'].nunique() - mba_df['InvoiceNo'].nunique())

Original cleaned rows: 524878
MBA modelling rows: 383555
MBA transactions: 19332
Removed transactions: 628


In [37]:
mba_basket_size = mba_df.groupby('InvoiceNo')['StockCode'].nunique()

print("Maximum basket size:", mba_basket_size.max())
print("Average basket size:", round(mba_basket_size.mean(), 2))
print("Median basket size:", mba_basket_size.median())

Maximum basket size: 100
Average basket size: 19.62
Median basket size: 15.0


In [38]:
# Look for possible non-product/service items

keywords = 'POSTAGE|MANUAL|BANK|CHARGE|AMAZON|CARRIAGE|DISCOUNT'

special_items = mba_df[
    mba_df['Description']
    .str.contains(keywords, case=False, na=False)
]

print("Potential non-product/service rows:",
      len(special_items))

special_items[
    ['StockCode', 'Description']
].drop_duplicates().sort_values('Description')

Potential non-product/service rows: 3198


,StockCode,Description
15017,AMAZONFEE,AMAZON FEE
3341,21518,BANK ACCOUNT GREETING CARD
44787,85168B,BLACK BAROQUE CARRIAGE CLOCK
4406,BANK CHARGES,Bank Charges
1423,C2,CARRIAGE
129,22644,CERAMIC CHERRY CAKE MONEY BANK
718,22645,CERAMIC HEART FAIRY CAKE MONEY BANK
29847,22647,CERAMIC LOVE HEART MONEY BANK
719,22650,CERAMIC PIRATE CHEST MONEY BANK
134,22646,CERAMIC STRAWBERRY CAKE MONEY BANK


In [39]:
service_codes = [
    'AMAZONFEE',
    'BANK CHARGES',
    'C2',
    'DOT',
    'M',
    'POST'
]

service_descriptions = [
    'AMAZON FEE',
    'BANK CHARGES',
    'CARRIAGE',
    'DOTCOM POSTAGE',
    'MANUAL',
    'NEXT DAY CARRIAGE',
    'PACKING CHARGE',
    'POSTAGE'
]

service_mask = (
    mba_df['StockCode'].astype(str).isin(service_codes)
    | mba_df['Description'].str.upper().isin(service_descriptions)
)

print("Service/adjustment rows to remove:", service_mask.sum())

Service/adjustment rows to remove: 1952


In [40]:
mba_df.loc[
    service_mask,
    ['StockCode', 'Description']
].drop_duplicates().sort_values(['StockCode', 'Description'])

,StockCode,Description
205748,23444,Next Day Carriage
252605,23574,PACKING CHARGE
15017,AMAZONFEE,AMAZON FEE
4406,BANK CHARGES,Bank Charges
1423,C2,CARRIAGE
5450,DOT,DOTCOM POSTAGE
2239,M,Manual
45,POST,POSTAGE
40383,m,Manual


In [41]:
product_df = mba_df[~service_mask].copy()

print("MBA rows before service filtering:", len(mba_df))
print("Product-only rows:", len(product_df))
print("Rows removed:", len(mba_df) - len(product_df))

MBA rows before service filtering: 383555
Product-only rows: 381603
Rows removed: 1952


In [42]:
remaining_services = product_df[
    product_df['StockCode'].astype(str).isin(service_codes)
    | product_df['Description'].str.upper().isin(service_descriptions)
]

print("Remaining service rows:", len(remaining_services))

Remaining service rows: 0


In [43]:
baskets = (
    product_df
    .groupby('InvoiceNo')['Description']
    .apply(list)
)

print("Number of baskets:", len(baskets))
print("\nFirst basket:")
print(baskets.iloc[0])

Number of baskets: 19148

First basket:
['WHITE HANGING HEART T-LIGHT HOLDER', 'WHITE METAL LANTERN', 'CREAM CUPID HEARTS COAT HANGER', 'KNITTED UNION FLAG HOT WATER BOTTLE', 'RED WOOLLY HOTTIE WHITE HEART.', 'SET 7 BABUSHKA NESTING BOXES', 'GLASS STAR FROSTED T-LIGHT HOLDER']


In [44]:
# Remove duplicate products within each basket
unique_baskets = baskets.apply(lambda x: list(set(x)))

print("Number of baskets:", len(unique_baskets))
print("First basket:")
print(unique_baskets.iloc[0])

Number of baskets: 19148
First basket:
['RED WOOLLY HOTTIE WHITE HEART.', 'WHITE METAL LANTERN', 'KNITTED UNION FLAG HOT WATER BOTTLE', 'CREAM CUPID HEARTS COAT HANGER', 'GLASS STAR FROSTED T-LIGHT HOLDER', 'WHITE HANGING HEART T-LIGHT HOLDER', 'SET 7 BABUSHKA NESTING BOXES']


In [46]:
%pip install mlxtend

  Using cached mlxtend-0.25.0-py3-none-any.whl.metadata (7.4 kB)
Using cached mlxtend-0.25.0-py3-none-any.whl (1.4 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import mlxtend

print("MLxtend version:", mlxtend.__version__)

MLxtend version: 0.25.0


In [3]:
# Remove duplicate products within each basket
unique_baskets = baskets.apply(lambda x: list(set(x)))

print("Number of baskets:", len(unique_baskets))
print("First basket:")
print(unique_baskets.iloc[0])

NameError: name 'baskets' is not defined

In [5]:
import pandas as pd

# Load our saved cleaned dataset
clean_df = pd.read_csv("../data/processed/cleaned_retail.csv")

print("Cleaned dataset loaded:", clean_df.shape)

# Calculate basket size
basket_size = clean_df.groupby('InvoiceNo')['StockCode'].nunique()

# Keep baskets with at most 100 unique products
valid_invoices = basket_size[basket_size <= 100].index

mba_df = clean_df[
    clean_df['InvoiceNo'].isin(valid_invoices)
].copy()

# Service / adjustment items identified earlier
service_codes = [
    'AMAZONFEE',
    'BANK CHARGES',
    'C2',
    'DOT',
    'M',
    'POST'
]

service_descriptions = [
    'AMAZON FEE',
    'BANK CHARGES',
    'CARRIAGE',
    'DOTCOM POSTAGE',
    'MANUAL',
    'NEXT DAY CARRIAGE',
    'PACKING CHARGE',
    'POSTAGE'
]

# Remove confirmed service/adjustment entries
service_mask = (
    mba_df['StockCode'].astype(str).isin(service_codes)
    | mba_df['Description'].str.upper().isin(service_descriptions)
)

product_df = mba_df[~service_mask].copy()

print("MBA rows:", len(mba_df))
print("Product-only rows:", len(product_df))
print("Product baskets:", product_df['InvoiceNo'].nunique())

C:\Users\Tarun Srinivasan\AppData\Local\Temp\ipykernel_1464\712527343.py:4: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  clean_df = pd.read_csv("../data/processed/cleaned_retail.csv")


Cleaned dataset loaded: (524878, 8)
MBA rows: 383555
Product-only rows: 381603
Product baskets: 19150


In [6]:
baskets = (
    product_df
    .groupby('InvoiceNo')['Description']
    .apply(list)
)

unique_baskets = baskets.apply(lambda x: list(set(x)))

print("Number of baskets:", len(unique_baskets))
print("First basket:")
print(unique_baskets.iloc[0])

Number of baskets: 19150
First basket:
['SET 7 BABUSHKA NESTING BOXES', 'GLASS STAR FROSTED T-LIGHT HOLDER', 'WHITE METAL LANTERN', 'CREAM CUPID HEARTS COAT HANGER', 'RED WOOLLY HOTTIE WHITE HEART.', 'WHITE HANGING HEART T-LIGHT HOLDER', 'KNITTED UNION FLAG HOT WATER BOTTLE']


In [7]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()

basket_array = te.fit(unique_baskets).transform(
    unique_baskets,
    sparse=True
)

basket_matrix = pd.DataFrame.sparse.from_spmatrix(
    basket_array,
    columns=te.columns_
)

print("Basket matrix shape:", basket_matrix.shape)

Basket matrix shape: (19150, 3884)


In [8]:
basket_matrix.head()

,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 DAISY PEGS IN WOOD BOX,12 EGG HOUSE PAINTED WOOD,12 HANGING EGGS HAND PAINTED,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE SKULL,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [9]:
from mlxtend.frequent_patterns import fpgrowth

frequent_itemsets = fpgrowth(
    basket_matrix,
    min_support=0.01,
    use_colnames=True
)

print("Number of frequent itemsets:", len(frequent_itemsets))

Number of frequent itemsets: 923


In [10]:
frequent_itemsets.sort_values(
    'support',
    ascending=False
).head(20)

,support,itemsets
0,0.101097,frozenset({WHITE HANGING HEART T-LIGHT HOLDER})
85,0.091175,frozenset({JUMBO BAG RED RETROSPOT})
248,0.090862,frozenset({REGENCY CAKESTAND 3 TIER})
409,0.075248,frozenset({PARTY BUNTING})
7,0.070392,frozenset({ASSORTED COLOUR BIRD ORNAMENT})
40,0.068721,frozenset({LUNCH BAG RED RETROSPOT})
438,0.058695,frozenset({SET OF 3 CAKE TINS PANTRY DESIGN})
154,0.055979,frozenset({LUNCH BAG BLACK SKULL.})
41,0.053890,frozenset({PACK OF 72 RETROSPOT CAKE CASES})
42,0.053786,frozenset({JUMBO BAG PINK POLKADOT})


In [11]:
frequent_itemsets['itemset_size'] = frequent_itemsets['itemsets'].apply(len)

print(
    frequent_itemsets['itemset_size']
    .value_counts()
    .sort_index()
)

itemset_size
1    576
2    272
3     73
4      2
Name: count, dtype: int64


In [12]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.30
)

print("Number of rules:", len(rules))

Number of rules: 669


In [13]:
rules[
    ['antecedents', 'consequents',
     'support', 'confidence', 'lift']
].sort_values(
    'lift',
    ascending=False
).head(20)

,antecedents,consequents,support,confidence,lift
645,frozenset({REGENCY SUGAR BOWL GREEN}),frozenset({REGENCY MILK JUG PINK}),0.010548,0.782946,56.578909
646,frozenset({REGENCY MILK JUG PINK}),frozenset({REGENCY SUGAR BOWL GREEN}),0.010548,0.762264,56.578909
648,frozenset({REGENCY TEA PLATE ROSES}),frozenset({REGENCY TEA PLATE GREEN}),0.011070,0.688312,52.099482
647,frozenset({REGENCY TEA PLATE GREEN}),frozenset({REGENCY TEA PLATE ROSES}),0.011070,0.837945,52.099482
441,frozenset({POPPY'S PLAYHOUSE LIVINGROOM}),frozenset({POPPY'S PLAYHOUSE BEDROOM}),0.010809,0.828000,49.090402
442,frozenset({POPPY'S PLAYHOUSE BEDROOM}),frozenset({POPPY'S PLAYHOUSE LIVINGROOM}),0.010809,0.640867,49.090402
444,frozenset({POPPY'S PLAYHOUSE KITCHEN}),frozenset({POPPY'S PLAYHOUSE LIVINGROOM}),0.010653,0.614458,47.067470
443,frozenset({POPPY'S PLAYHOUSE LIVINGROOM}),frozenset({POPPY'S PLAYHOUSE KITCHEN}),0.010653,0.816000,47.067470
558,frozenset({CHILDRENS CUTLERY DOLLY GIRL}),frozenset({CHILDRENS CUTLERY SPACEBOY}),0.010496,0.761364,46.433483
559,frozenset({CHILDRENS CUTLERY SPACEBOY}),frozenset({CHILDRENS CUTLERY DOLLY GIRL}),0.010496,0.640127,46.433483


In [14]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.30
)

print("Number of rules:", len(rules))

Number of rules: 669


In [15]:
rules[
    ['antecedents', 'consequents',
     'support', 'confidence', 'lift']
].sort_values(
    'lift',
    ascending=False
).head(20)

,antecedents,consequents,support,confidence,lift
645,frozenset({REGENCY SUGAR BOWL GREEN}),frozenset({REGENCY MILK JUG PINK}),0.010548,0.782946,56.578909
646,frozenset({REGENCY MILK JUG PINK}),frozenset({REGENCY SUGAR BOWL GREEN}),0.010548,0.762264,56.578909
648,frozenset({REGENCY TEA PLATE ROSES}),frozenset({REGENCY TEA PLATE GREEN}),0.011070,0.688312,52.099482
647,frozenset({REGENCY TEA PLATE GREEN}),frozenset({REGENCY TEA PLATE ROSES}),0.011070,0.837945,52.099482
441,frozenset({POPPY'S PLAYHOUSE LIVINGROOM}),frozenset({POPPY'S PLAYHOUSE BEDROOM}),0.010809,0.828000,49.090402
442,frozenset({POPPY'S PLAYHOUSE BEDROOM}),frozenset({POPPY'S PLAYHOUSE LIVINGROOM}),0.010809,0.640867,49.090402
444,frozenset({POPPY'S PLAYHOUSE KITCHEN}),frozenset({POPPY'S PLAYHOUSE LIVINGROOM}),0.010653,0.614458,47.067470
443,frozenset({POPPY'S PLAYHOUSE LIVINGROOM}),frozenset({POPPY'S PLAYHOUSE KITCHEN}),0.010653,0.816000,47.067470
558,frozenset({CHILDRENS CUTLERY DOLLY GIRL}),frozenset({CHILDRENS CUTLERY SPACEBOY}),0.010496,0.761364,46.433483
559,frozenset({CHILDRENS CUTLERY SPACEBOY}),frozenset({CHILDRENS CUTLERY DOLLY GIRL}),0.010496,0.640127,46.433483


In [16]:
rules_display = rules.copy()

rules_display['antecedent'] = rules_display['antecedents'].apply(
    lambda x: ', '.join(x)
)

rules_display['consequent'] = rules_display['consequents'].apply(
    lambda x: ', '.join(x)
)

rules_display = rules_display[
    ['antecedent', 'consequent',
     'support', 'confidence', 'lift']
]

rules_display = rules_display.sort_values(
    'lift',
    ascending=False
)

rules_display.head(20)

,antecedent,consequent,support,confidence,lift
645,REGENCY SUGAR BOWL GREEN,REGENCY MILK JUG PINK,0.010548,0.782946,56.578909
646,REGENCY MILK JUG PINK,REGENCY SUGAR BOWL GREEN,0.010548,0.762264,56.578909
648,REGENCY TEA PLATE ROSES,REGENCY TEA PLATE GREEN,0.011070,0.688312,52.099482
647,REGENCY TEA PLATE GREEN,REGENCY TEA PLATE ROSES,0.011070,0.837945,52.099482
441,POPPY'S PLAYHOUSE LIVINGROOM,POPPY'S PLAYHOUSE BEDROOM,0.010809,0.828000,49.090402
442,POPPY'S PLAYHOUSE BEDROOM,POPPY'S PLAYHOUSE LIVINGROOM,0.010809,0.640867,49.090402
444,POPPY'S PLAYHOUSE KITCHEN,POPPY'S PLAYHOUSE LIVINGROOM,0.010653,0.614458,47.067470
443,POPPY'S PLAYHOUSE LIVINGROOM,POPPY'S PLAYHOUSE KITCHEN,0.010653,0.816000,47.067470
558,CHILDRENS CUTLERY DOLLY GIRL,CHILDRENS CUTLERY SPACEBOY,0.010496,0.761364,46.433483
559,CHILDRENS CUTLERY SPACEBOY,CHILDRENS CUTLERY DOLLY GIRL,0.010496,0.640127,46.433483


In [17]:
def recommend_products(product, rules_df, top_n=5):
    product = product.upper()

    matches = rules_df[
        rules_df['antecedent'].str.upper().str.contains(product, na=False)
    ].copy()

    matches = matches.sort_values(
        ['lift', 'confidence', 'support'],
        ascending=False
    )

    return matches[
        ['antecedent', 'consequent', 'support', 'confidence', 'lift']
    ].head(top_n)

In [19]:
recommend_products(
    'WHITE HANGING HEART T-LIGHT HOLDER',
    rules_display,
    top_n=5
)

,antecedent,consequent,support,confidence,lift


In [20]:
def get_recommendations(product, rules_df, top_n=5):
    product = product.upper()

    matches = rules_df[
        rules_df['antecedent'].str.upper().str.contains(product, na=False)
    ].copy()

    matches = matches.sort_values(
        ['lift', 'confidence'],
        ascending=False
    ).head(top_n)

    if matches.empty:
        print("No recommendations found for:", product)
        return

    print(f"\nRecommendations for: {product}\n")

    for i, row in enumerate(matches.itertuples(), 1):
        print(f"{i}. {row.consequent}")
        print(f"   Confidence: {row.confidence:.2%}")
        print(f"   Lift: {row.lift:.2f}")
        print()

In [21]:
print("Total rules:", len(rules_display))

print("\nProducts appearing in antecedents:")
print(rules_display['antecedent'].head(20).to_string(index=False))

Total rules: 669

Products appearing in antecedents:
          REGENCY SUGAR BOWL GREEN
             REGENCY MILK JUG PINK
           REGENCY TEA PLATE ROSES
           REGENCY TEA PLATE GREEN
      POPPY'S PLAYHOUSE LIVINGROOM
         POPPY'S PLAYHOUSE BEDROOM
         POPPY'S PLAYHOUSE KITCHEN
      POPPY'S PLAYHOUSE LIVINGROOM
      CHILDRENS CUTLERY DOLLY GIRL
        CHILDRENS CUTLERY SPACEBOY
         POPPY'S PLAYHOUSE BEDROOM
         POPPY'S PLAYHOUSE KITCHEN
     SET/6 RED SPOTTY PAPER PLATES
       SET/6 RED SPOTTY PAPER CUPS
SMALL DOLLY MIX DESIGN ORANGE BOWL
      SMALL MARSHMALLOWS PINK BOWL
   CHILDRENS CUTLERY POLKADOT BLUE
   CHILDRENS CUTLERY POLKADOT PINK
WOODEN TREE CHRISTMAS SCANDINAVIAN
WOODEN STAR CHRISTMAS SCANDINAVIAN


In [22]:
test_product = rules_display.iloc[0]['antecedent']

print("Testing product/rule:")
print(test_product)

Testing product/rule:
REGENCY SUGAR BOWL GREEN


In [23]:
get_recommendations(
    test_product,
    rules_display,
    top_n=5
)


Recommendations for: REGENCY SUGAR BOWL GREEN

1. REGENCY MILK JUG PINK
   Confidence: 78.29%
   Lift: 56.58



In [24]:
recommendation_table = rules_display[
    ['antecedent', 'consequent', 'support', 'confidence', 'lift']
].copy()

recommendation_table['confidence'] = (
    recommendation_table['confidence'] * 100
).round(2)

recommendation_table['lift'] = (
    recommendation_table['lift']
).round(2)

recommendation_table['support'] = (
    recommendation_table['support'] * 100
).round(2)

recommendation_table.head(10)

,antecedent,consequent,support,confidence,lift
645,REGENCY SUGAR BOWL GREEN,REGENCY MILK JUG PINK,1.05,78.29,56.58
646,REGENCY MILK JUG PINK,REGENCY SUGAR BOWL GREEN,1.05,76.23,56.58
648,REGENCY TEA PLATE ROSES,REGENCY TEA PLATE GREEN,1.11,68.83,52.10
647,REGENCY TEA PLATE GREEN,REGENCY TEA PLATE ROSES,1.11,83.79,52.10
441,POPPY'S PLAYHOUSE LIVINGROOM,POPPY'S PLAYHOUSE BEDROOM,1.08,82.80,49.09
442,POPPY'S PLAYHOUSE BEDROOM,POPPY'S PLAYHOUSE LIVINGROOM,1.08,64.09,49.09
444,POPPY'S PLAYHOUSE KITCHEN,POPPY'S PLAYHOUSE LIVINGROOM,1.07,61.45,47.07
443,POPPY'S PLAYHOUSE LIVINGROOM,POPPY'S PLAYHOUSE KITCHEN,1.07,81.60,47.07
558,CHILDRENS CUTLERY DOLLY GIRL,CHILDRENS CUTLERY SPACEBOY,1.05,76.14,46.43
559,CHILDRENS CUTLERY SPACEBOY,CHILDRENS CUTLERY DOLLY GIRL,1.05,64.01,46.43


In [25]:
recommendation_table.to_csv(
    "../models/recommendation_rules.csv",
    index=False
)

print("Recommendation rules saved!")

Recommendation rules saved!
